# 03 - English Speech Synthesis with Voice Cloning (F5-TTS)

Uses **F5-TTS** to generate English speech from translated segments,
cloning the original speaker's voice using diarized voice references.

**Requirements:** Google Colab with GPU runtime (T4 or better).

**Inputs on Drive:**
- `data/translations/*_en.json` (with speaker labels from diarization)
- `data/audio/voice_reference/SPEAKER_00_ref.wav` and `SPEAKER_01_ref.wav`

**Output:** Per-segment WAV files in `data/tts/<video>_segments/`

In [ ]:
# Cell 1: Install dependencies
!pip install -q f5-tts soundfile numpy tqdm

In [ ]:
# Cell 2: Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import shutil
import soundfile as sf
import numpy as np
from tqdm import tqdm

PROJECT_DIR = '/content/drive/MyDrive/AIVideoLanguageTransformation'
TRANSLATIONS_DIR = os.path.join(PROJECT_DIR, 'data', 'translations')
VOICE_REF_DIR = os.path.join(PROJECT_DIR, 'data', 'audio', 'voice_reference')
TTS_DIR = os.path.join(PROJECT_DIR, 'data', 'tts')

print(f'Translations: {sorted(os.listdir(TRANSLATIONS_DIR)) if os.path.exists(TRANSLATIONS_DIR) else "NOT FOUND"}')
print(f'Voice refs: {sorted(os.listdir(VOICE_REF_DIR)) if os.path.exists(VOICE_REF_DIR) else "NOT FOUND"}')

In [ ]:
# Cell 3: Trim voice references to ~8s (F5-TTS can overflow with longer refs)
TRIMMED_REF_DIR = '/content/trimmed_voice_refs'
os.makedirs(TRIMMED_REF_DIR, exist_ok=True)

MAX_REF_SECONDS = 8

for f in sorted(os.listdir(VOICE_REF_DIR)):
    if not f.endswith('.wav') or not f.startswith('SPEAKER_'):
        continue
    data, sr = sf.read(os.path.join(VOICE_REF_DIR, f))
    duration = len(data) / sr
    if duration > MAX_REF_SECONDS:
        data = data[:int(MAX_REF_SECONDS * sr)]
        print(f'{f}: trimmed {duration:.1f}s -> {MAX_REF_SECONDS}s')
    else:
        print(f'{f}: {duration:.1f}s (OK)')
    sf.write(os.path.join(TRIMMED_REF_DIR, f), data, sr)

VOICE_REF_DIR = TRIMMED_REF_DIR
print(f'\nUsing trimmed refs from: {TRIMMED_REF_DIR}')

In [ ]:
# Cell 4: Load F5-TTS model
from f5_tts.api import F5TTS

print('Loading F5-TTS model...')
tts = F5TTS()
print('Model loaded.')

In [ ]:
# Cell 5: Define helper functions

def load_voice_refs(voice_ref_dir):
    """Load SPEAKER_* voice reference files from diarization."""
    refs = {}
    for f in sorted(os.listdir(voice_ref_dir)):
        if not f.endswith('.wav') or not f.startswith('SPEAKER_'):
            continue
        key = f.replace('_ref.wav', '').replace('_ref', '')
        refs[key] = os.path.join(voice_ref_dir, f)
    print(f'Voice references: {refs}')
    return refs


def get_voice_ref(seg, voice_refs, default_ref):
    """Match segment to correct speaker's voice reference."""
    speaker = seg.get('speaker', '')
    if speaker in voice_refs:
        return voice_refs[speaker]
    for key, path in voice_refs.items():
        if key.lower() in speaker.lower() or speaker.lower() in key.lower():
            return path
    return default_ref


def synthesize_segments(translation_path, voice_refs, default_ref, output_dir):
    """Synthesize English speech for all segments using per-speaker voice refs."""
    os.makedirs(output_dir, exist_ok=True)

    with open(translation_path, 'r', encoding='utf-8') as f:
        segments = json.load(f)

    print(f'Synthesizing {len(segments)} segments...')

    for seg in tqdm(segments):
        text = seg.get('text_en', seg.get('text_en_deepl', ''))
        if not text.strip():
            continue

        output_path = os.path.join(output_dir, f"segment_{seg['id']:04d}.wav")

        # Skip if already exists (resumability)
        if os.path.exists(output_path):
            audio_data, sr = sf.read(output_path)
            seg['tts_file'] = os.path.basename(output_path)
            seg['tts_duration'] = round(len(audio_data) / sr, 3)
            continue

        ref_path = get_voice_ref(seg, voice_refs, default_ref)

        try:
            wav, sr, _ = tts.infer(
                ref_file=ref_path,
                ref_text='',
                gen_text=text,
            )
            sf.write(output_path, wav, sr)
            seg['tts_file'] = os.path.basename(output_path)
            seg['tts_duration'] = round(len(wav) / sr, 3)
        except Exception as e:
            print(f"  Error on segment {seg['id']}: {e}")
            seg['tts_file'] = None
            seg['tts_duration'] = 0

    # Save updated translation JSON with TTS metadata
    with open(translation_path, 'w', encoding='utf-8') as f:
        json.dump(segments, f, ensure_ascii=False, indent=2)

    return segments

In [ ]:
# Cell 6: Delete old segments and run synthesis
voice_refs = load_voice_refs(VOICE_REF_DIR)
ref_files = sorted([f for f in os.listdir(VOICE_REF_DIR) if f.endswith('.wav') and f.startswith('SPEAKER_')])
default_ref = os.path.join(VOICE_REF_DIR, ref_files[0]) if ref_files else None

translation_files = sorted([f for f in os.listdir(TRANSLATIONS_DIR) if f.endswith('_en.json')])
print(f'Translation files: {translation_files}')

# Delete old segments
for trans_file in translation_files:
    stem = trans_file.replace('_en.json', '')
    old_dir = os.path.join(TTS_DIR, f'{stem}_segments')
    if os.path.exists(old_dir):
        count = len([f for f in os.listdir(old_dir) if f.endswith('.wav')])
        print(f'Removing {count} old segments from {old_dir}')
        shutil.rmtree(old_dir)

# Synthesize
for trans_file in translation_files:
    stem = trans_file.replace('_en.json', '')
    trans_path = os.path.join(TRANSLATIONS_DIR, trans_file)
    output_dir = os.path.join(TTS_DIR, f'{stem}_segments')

    print(f'\n{"="*60}')
    print(f'Video: {stem}')
    synthesize_segments(trans_path, voice_refs, default_ref, output_dir)

print('\nAll synthesis complete!')

In [ ]:
# Cell 7: Listen to a sample
from IPython.display import Audio

sample_dir = os.path.join(TTS_DIR, sorted(os.listdir(TTS_DIR))[0])
sample_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.wav')])
if sample_files:
    sample_path = os.path.join(sample_dir, sample_files[0])
    print(f'Playing: {sample_path}')
    Audio(sample_path)